# **LangChain Memory**

Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At bare minimum, a conversational system should be able to access some window of past messages directly. A more complex system will need to have a world model that it is constantly updating, which allows it to do things like maintain information about entities and their relationships.

We call this ability to store information about past interactions "memory". LangChain provides a lot of utilities for adding memory to a system. These utilities can be used by themselves or incorporated seamlessly into a chain.

## **Building memory into a system**
The two core design decisions in any memory system are:
- How state is stored
- How state is queried

## **What's covered?**
- ConversationBufferMemory
- End-to-end Example: Conversational AI Bot
- Saving and Loading a Chat History
- ConversationBufferWindowMemory

## **ConversationBufferMemory**

Let's take a look at how to use ConversationBufferMemory in chains. ConversationBufferMemory is an extremely simple form of memory that just keeps a list of chat messages in a buffer and passes those into the prompt template.

This memory type can be connected to a conversation. It allows for storing messages and then extracts the messages in a variable.

In [1]:
# Importing ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

# Set up an instance of Conversation buffer memory
memory = ConversationBufferMemory()

C:\Users\KUMAR SUNDRAM\AppData\Local\Temp\ipykernel_15712\499927444.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [3]:
# Loading Memory

memory.load_memory_variables({})

{'history': ''}

In [5]:
# Loading memory buffer with history

memory.buffer

''

### **Renaming the memory variable and Adding Messages**

In [7]:
memory = ConversationBufferMemory(memory_key="chat_history")

memory.chat_memory.add_user_message("hi!")
memory.chat_memory.add_ai_message("what's up?")

In [9]:
memory.load_memory_variables({})

{'chat_history': "Human: hi!\nAI: what's up?"}

In [11]:
memory.buffer

"Human: hi!\nAI: what's up?"

### **By default, memory is returned as a single string. In order to return as a list of messages, you can set `return_messages=True`**

In [13]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

memory.chat_memory.add_user_message("hi!")
memory.chat_memory.add_ai_message("what's up?")

In [15]:
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
  AIMessage(content="what's up?", additional_kwargs={}, response_metadata={})]}

In [17]:
memory.buffer

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content="what's up?", additional_kwargs={}, response_metadata={})]

## **End-to-end Example: Conversational AI Bot**

<img src="images/memory.png">

### **Steps:**
1. Import Chat Model and Configure the API Key
2. Create Chat Template
3. Initialize the Memory
4. Create a Output Parser
5. Build a Chain
6. Invoke the chain with human_input and chat_history
7. Saving to memory
8. Run Step 6 and 7 in a loop

In [19]:
# Step 1 - Import Chat Model and Configure the API Key

from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

# Set the OpenAI Key and initialize a ChatModel
chat_model = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

In [21]:
# Step 2 - Create Chat Template

from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        # The persistent system prompt
        SystemMessage(
            content="You are a chatbot having a conversation with a human."
        ),
        # Creating a chat_history placeholder
        MessagesPlaceholder(
            variable_name="chat_history"
        ),  
        # Human Prompt
        HumanMessagePromptTemplate.from_template(
            "{human_input}"
        ),
    ]
)

In [23]:
# Step 3 - Initialize the Memory

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [25]:
# Step 4 - Create a Output Parser

from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

#### **RunnablePassthrough:** RunnablePassthrough on its own allows you to pass inputs unchanged.

In [27]:
# Step 5 - Build a Chain (Another way)
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Define a function to load the message from memory
def get_messages_from_memory(human_input):
    return memory.load_memory_variables(human_input)['chat_history']

# Define a chain
chain = RunnablePassthrough.assign(
            chat_history=RunnableLambda(get_messages_from_memory)
        ) | chat_prompt_template | chat_model | output_parser

In [29]:
# Step 6 - Invoke the chain with human_input and chat_history

query = {"human_input": "Hi, How are you?"}

response = chain.invoke(query)

response

"Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?"

In [31]:
memory.buffer

[]

In [33]:
# Step 7 - Saving to memory

memory.save_context(query, {"output" : response})

memory.buffer

[HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={}, response_metadata={})]

In [35]:
# Step 8 - Run Step 6 and 7 in a loop

while True:
    query = {"human_input" : input('Enter your input: ')}
    print(f"*User: {query['human_input']}")
    if query["human_input"] in ['bye', 'quit', 'exit']:
        break
    response = chain.invoke(query)
    print(f"*AI: {response}")

    memory.save_context(query, {"output" : response})

Enter your input:  hi, how are you?


*User: hi, how are you?
*AI: Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?


Enter your input:  what is KNN? please explain in simple language with step by step


*User: what is KNN? please explain in simple language with step by step
*AI: K-Nearest Neighbors (KNN) is a simple and easy-to-understand machine learning algorithm used for classification and regression tasks. Here's a step-by-step explanation of how KNN works:

1. **Step 1: Choose the value of K**: K is the number of nearest neighbors to consider when making a prediction. You need to choose a suitable value for K, which is usually an odd number to prevent ties.

2. **Step 2: Calculate the distance**: KNN calculates the distance between the new data point and all other data points in the dataset. The most commonly used distance metric is Euclidean distance, but other metrics like Manhattan or Minkowski can also be used.

3. **Step 3: Find the K nearest neighbors**: Once the distances are calculated, KNN finds the K data points in the training set that are closest to the new data point based on the calculated distances.

4. **Step 4: Make a prediction**: For classification tasks, KNN t

Enter your input:  what is MULTILAYER PERCEPTRON MODEL?


*User: what is MULTILAYER PERCEPTRON MODEL?
*AI: A Multilayer Perceptron (MLP) is a type of artificial neural network that consists of multiple layers of nodes, or neurons, arranged in a feedforward manner. Here's an explanation of the Multilayer Perceptron model:

1. **Input Layer**: The input layer receives the input data and passes it on to the next layer. Each node in the input layer represents a feature of the input data.

2. **Hidden Layers**: Between the input and output layers, there can be one or more hidden layers in an MLP. Each hidden layer consists of multiple neurons that perform computations on the input data using activation functions.

3. **Output Layer**: The output layer produces the final result or prediction based on the computations performed in the hidden layers. The number of nodes in the output layer depends on the type of problem being solved (e.g., binary classification, multi-class classification, regression).

4. **Activation Function**: Each neuron in the 

Enter your input:  exit


*User: exit


In [37]:
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='hi, how are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='what is KNN? please explain in simple language with step by step', additional_kwargs={}, response_metadata={}),
  AIMessage(content="K-Nearest Neighbors (KNN) is a simple and easy-to-understand machine learning algorithm used for classification and regression tasks. Here's a step-by-step explanation of how KNN works:\n\n1. **Step 1: Choose the value of K**: K is the number of nearest neighbors to consider when ma

## **Saving a Chat History**

**Let's now learn to save this history on the disk so that whenever we can load the history whenever we chat with our assistant.**

In [39]:
import pickle

chat_history = pickle.dumps(memory)

with open("chats_data/conversation_memory.pkl", "wb") as f:
    f.write(chat_history)

## **Loading a Chat History**

In [41]:
chat_history_loaded = pickle.load(open("chats_data/conversation_memory.pkl", "rb"))

print(chat_history_loaded)

chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={}, response_metadata={}), HumanMessage(content='hi, how are you?', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={}, response_metadata={}), HumanMessage(content='what is KNN? please explain in simple language with step by step', additional_kwargs={}, response_metadata={}), AIMessage(content="K-Nearest Neighbors (KNN) is a simple and easy-to-understand machine learning algorithm used for classification and regression tasks. Here's a step-by-step explanation of how KNN works:\n\n1. **Step 1: Choose the value of K**: K is the number of nearest neighbor

In [50]:
chat_history_loaded.load_memory_variables({})

{'chat_history': [HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello! I'm just a chatbot, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Hi, how are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'm just a chatbot, so I don't have feelings, but I'm here and ready to assist you. How can I help you today?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='what is CNN?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='CNN stands for Cable News Network. It is an American news-based pay television channel owned by WarnerMedia News & Sports. CNN was founded in 1980 by media mogul Ted Turner and is one of the most widely watched news channels in the world. It provides 24-hour news coverage and broadcasts a variety of programs on current events, politics, b

## **ConversationBufferWindowMemory [Beta]**

With this we will have an ability to call back just a window of conversation.

`ConversationBufferWindowMemory` keeps a list of the interactions of the conversation over time. It only uses the last K interactions. This can be useful for keeping a sliding window of the most recent interactions, so the buffer does not get too large.

In [53]:
from langchain_openai import ChatOpenAI

# Set the OpenAI Key and initialize a ChatModel
chat_model = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

In [55]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

chat_prompt_template = ChatPromptTemplate.from_messages(
    [
        # The persistent system prompt
        SystemMessage(
            content="You are a chatbot having a conversation with a human."
        ),
        # Creating a chat_history placeholder
        MessagesPlaceholder(
            variable_name="chat_history"
        ),  
        # Human Prompt
        HumanMessagePromptTemplate.from_template(
            "{human_input}"
        ),
    ]
)

In [57]:
from langchain.memory import ConversationBufferWindowMemory

# Set up an instance of Conversation buffer window memory
window_memory = ConversationBufferWindowMemory(memory_key="chat_history", return_messages=True, k=2)
# here, k most recent interactions will be refered to answer the user queries

C:\Users\KUMAR SUNDRAM\AppData\Local\Temp\ipykernel_24456\384106824.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(memory_key="chat_history", return_messages=True, k=2)


In [59]:
window_memory.load_memory_variables({})

{'chat_history': []}

In [61]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Define a function to load the message from memory
def get_messages_from_memory(human_input):
    return window_memory.load_memory_variables(human_input)['chat_history']

# Define a chain
chain = RunnablePassthrough.assign(
            chat_history=RunnableLambda(get_messages_from_memory)
        ) | chat_prompt_template | chat_model

In [63]:
query = {"human_input" : "What are the top 3 states of India on the basis of population?"}

response = chain.invoke(query)

window_memory.save_context(query, {"output" : response.content})

print(response.content)

The top 3 states in India based on population are Uttar Pradesh, Maharashtra, and Bihar.


In [64]:
window_memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='What are the top 3 states of India on the basis of population?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The top 3 states in India based on population are Uttar Pradesh, Maharashtra, and Bihar.', additional_kwargs={}, response_metadata={})]}

In [67]:
query = {"human_input" : "What are the next 2 states?"}

response = chain.invoke(query)

window_memory.save_context(query, {"output" : response.content})

print(response.content)

The next two states in India based on population are West Bengal and Madhya Pradesh.


In [68]:
window_memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='What are the top 3 states of India on the basis of population?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The top 3 states in India based on population are Uttar Pradesh, Maharashtra, and Bihar.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What are the next 2 states?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The next two states in India based on population are West Bengal and Madhya Pradesh.', additional_kwargs={}, response_metadata={})]}

## **SQLChatMessageHistory**

`ChatMessageHistory` allows us to store separate conversation histories per user or session which is often done by the real-time chatbots. `session_id` is used to distinguish between separate conversations.

In order to use it, we can use a `get_session_history` function which take `session_id` and returns a message history object.

There is a support of many `Memory` components under `langchain_community.chat_message_histories`, like:
1. AstraDBChatMessageHistory
2. DynamoDBChatMessageHistory
3. CassandraChatMessageHistory
4. ElasticsearchChatMessageHistory
5. KafkaChatMessageHistory
6. MongoDBChatMessageHistory
7. RedisChatMessageHistory
8. PostgresChatMessageHistory
9. SQLChatMessageHistory

**[Click Here](https://python.langchain.com/v0.2/docs/integrations/memory/)** to read more.

### **Usage**

To use the storage you need to provide only 2 things:

1. Session Id - a unique identifier of the session, like user name, email, chat id etc.
2. Connection string
    - For SQL (SQLAlchemy) - A string that specifies the database connection. It will be passed to SQLAlchemy create_engine function.
    - For SQLite - A string that specifies the database connection. For SQLite, that string is slqlite:/// followed by the name of the database file. If that file doesn't exist, it will be created.

In [72]:
# Set the OpenAI Key and initialize a ChatModel

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

In [74]:
# Create a connection with the database and 
# return the chat message history for a session id

from langchain_community.chat_message_histories import SQLChatMessageHistory


def get_session_message_history_from_db(session_id):
    chat_message_history = SQLChatMessageHistory(
                                   session_id=session_id, 
                                   connection="sqlite:///chats_data/sqlite.db"
                               )
    return chat_message_history

In [75]:
# Create a chat template

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_template = ChatPromptTemplate.from_messages([
                    ("system", "You are a helpful AI assistant."), 
                    MessagesPlaceholder(variable_name="history"), 
                    ("human", "{human_input}")
                ])

In [78]:
# Defining the chain

chain = chat_template | chat_model

In [80]:
# Use RunnableWithMessageHistory to load 
from langchain_core.runnables.history import RunnableWithMessageHistory

conversation_chain = RunnableWithMessageHistory(
                        chain, 
                        get_session_message_history_from_db,
                        input_messages_key="human_input", 
                        history_messages_key="history"
                    )

In [92]:
# This is where we configure the session id
user_id = "sundram"
config = {"configurable": {"session_id": user_id}}

input_prompt = {"human_input": "My name is sundram. Can you tell me the capital of Himachal?"}
response = conversation_chain.invoke(input_prompt, config=config)

response.content

'The capital of Himachal Pradesh is Shimla.'

In [94]:
# This is where we configure the session id
user_id = "kumar"
config = {"configurable": {"session_id": user_id}}

input_prompt = {"human_input": "My name is Kumar Sundram. What is the biggest state in India?"}
response = conversation_chain.invoke(input_prompt, config=config)

response.content

'The biggest state in India by area is Rajasthan.'

In [96]:
def chat_bot(session_id, prompt):
    config = {"configurable": {"session_id": user_id}}
    input_prompt = {"human_input": prompt}

    response = conversation_chain.invoke(input_prompt, config=config)

    return response.content

In [98]:
user_id = "kumar"
input_prompt = "Do you remember my name?"
chat_bot(session_id=user_id, prompt=input_prompt)

'Yes, your name is Kumar Sundram.'

In [100]:
user_id = "sundram"
input_prompt = "Do you remember my name?"
chat_bot(session_id=user_id, prompt=input_prompt)

'Yes, your name is Sundram. How can I assist you further, Sundram?'